# Zadanie 2: RAG na własnym korpusie

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adrianstando/2026Z-PJATK-TEG/blob/main/02-rag/zadanie.ipynb)

**Oddanie:** mail z tematem `[TEG] lab-02`, w załączniku ten notebook **z wynikami komórek**, bez kluczy API.

**Termin:** przed zajęciami 3.

Części:
1. Korpus z metadanymi, chunking i Chroma z filtrem.
2. Wyszukiwanie: wektory, BM25, hybryda, reranking i porównanie na zestawie testowym.
3. Pipeline RAG i ocena odpowiedzi przez LLM.

Kod z [notebooka z zajęć](notebook.ipynb) można kopiować i modyfikować.

## 0. Konfiguracja

In [ ]:
# Colab / Deepnote / Codespaces: odkomentuj i uruchom raz
# %pip install -q openai python-dotenv numpy chromadb rank-bm25 langchain-text-splitters sentence-transformers

Dostawcę wybiera się zmienną `LLM_PROVIDER` (plik `.env` albo w Colabie: ikona klucza → *Secrets*). Instrukcja konfiguracji dostępu: [00-dostep-do-llm](../00-dostep-do-llm/README.md).

| `LLM_PROVIDER` | Wymagania |
|---|---|
| `azure` | `AZURE_OPENAI_ENDPOINT`, `AZURE_OPENAI_API_KEY`, deploymenty `gpt-4o-mini` i `text-embedding-3-small` |
| `openai` | `OPENAI_API_KEY` |
| `deepseek` | `DEEPSEEK_API_KEY`; tanie API, bez embeddingów |
| `ollama` | [Ollama](https://ollama.com) lokalnie albo w Colabie na darmowym GPU (komórka niżej), modele `gemma4:e4b`, `gemma3:1b`, `embeddinggemma` |
| `proxy` | uruchomione [proxy dla subskrypcji](../tools/teg-proxy/README.md) (Copilot, Claude, Codex); bez embeddingów |

Wszyscy dostawcy udostępniają API zgodne z OpenAI, więc dalszy kod jest wspólny. Gdy dostawca nie liczy embeddingów, służy do tego `EMBED_PROVIDER` (domyślnie `ollama`).

### Ollama w Colabie (darmowe GPU)

Colab daje za darmo kartę graficzną NVIDIA T4, na której działa Ollama. Najpierw *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → T4 GPU*, potem komórka poniżej (odkomentowana). Na początek wystarczy mały `gemma3:1b`; większy `gemma4:e4b` zajmuje ok. 10 GB. Bez niego w *Secrets* ustawia się `OLLAMA_CHAT=gemma3:1b`.

In [ ]:
# Colab z GPU: odkomentuj i uruchom raz na sesję
# !nvidia-smi                                   # czy GPU jest włączone
# !sudo apt-get -qq install -y zstd pciutils > /dev/null
# !curl -fsSL https://ollama.com/install.sh | sh
# import subprocess
# subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# !ollama pull gemma3:1b
# !ollama pull embeddinggemma
# !ollama pull gemma4:e4b                       # opcjonalnie, ~10 GB

In [ ]:
import os

try:  # Colab: sekrety z panelu "Secrets"
    from google.colab import userdata
    for k in ["LLM_PROVIDER", "EMBED_PROVIDER", "OLLAMA_CHAT", "OPENAI_API_KEY", "AZURE_OPENAI_ENDPOINT",
              "AZURE_OPENAI_API_KEY", "DEEPSEEK_API_KEY", "ANTHROPIC_API_KEY"]:
        try:
            os.environ.setdefault(k, userdata.get(k))
        except Exception:
            pass
except ImportError:  # lokalnie: plik .env
    from dotenv import load_dotenv
    load_dotenv(override=True)

from openai import OpenAI, AzureOpenAI

# chat: model rozumujący, classic: model bez rozumowania, embed: model embeddingów.
# W Azure to NAZWY DEPLOYMENTÓW utworzonych w AI Foundry.
MODELS = {
    "openai":   {"chat": "gpt-5-nano", "classic": "gpt-4.1-mini", "embed": "text-embedding-3-small"},
    "azure":    {"chat": "gpt-4o-mini", "classic": "gpt-4o-mini", "embed": "text-embedding-3-small"},
    "deepseek": {"chat": "deepseek-reasoner", "classic": "deepseek-chat", "embed": None},
    "ollama":   {"chat": os.getenv("OLLAMA_CHAT", "gemma4:e4b"), "classic": "gemma3:1b", "embed": "embeddinggemma"},
    "proxy":    {"chat": "copilot/gpt-5-mini", "classic": "copilot/gpt-5-mini", "embed": None},
}

def make_client(provider):
    if provider == "azure":
        return AzureOpenAI(api_version="2024-12-01-preview")  # AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
    if provider == "deepseek":
        return OpenAI(base_url="https://api.deepseek.com", api_key=os.environ["DEEPSEEK_API_KEY"])
    if provider == "ollama":
        return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # klucz wymagany, ale ignorowany
    if provider == "proxy":
        return OpenAI(base_url="http://localhost:8787/v1", api_key="teg")
    return OpenAI()  # OPENAI_API_KEY

PROVIDER = os.getenv("LLM_PROVIDER", "ollama")
EMBED_PROVIDER = os.getenv("EMBED_PROVIDER") or (PROVIDER if MODELS[PROVIDER]["embed"] else "ollama")

client = make_client(PROVIDER)
embed_client = client if EMBED_PROVIDER == PROVIDER else make_client(EMBED_PROVIDER)
CHAT, CLASSIC = MODELS[PROVIDER]["chat"], MODELS[PROVIDER]["classic"]
EMBED = MODELS[EMBED_PROVIDER]["embed"]
print(f"czat: {PROVIDER} ({CHAT}, klasyczny: {CLASSIC}) | embeddingi: {EMBED_PROVIDER} ({EMBED})")

## Część 1. Korpus, chunking, Chroma

1. Korpus: **co najmniej 10 dokumentów** (każdy min. 400 znaków) z wybranej dziedziny, np. regulamin studiów, instrukcje sprzętu, przepisy kulinarne, dokumentacja biblioteki. Można też użyć korpusu z zajęć (`data/`) i dopisać do niego **5 własnych dokumentów**. Każdy dokument ma co najmniej jedno pole metadanych, po którym ma sens filtrować (kategoria, rok, wersja, język).
2. Chunking `RecursiveCharacterTextSplitter` z wybranym `chunk_size` i `chunk_overlap`; wypisanie liczby chunków i jednego przykładowego chunku z metadanymi.
3. Kolekcja Chroma i **jedno zapytanie**, dla którego wynik z filtrem `where` różni się od wyniku bez filtra. Wypisanie obu list.

In [ ]:
# TODO: wczytanie albo zdefiniowanie dokumentów z metadanymi

In [ ]:
# TODO: chunking, kolekcja Chroma, zapytanie bez filtra i z filtrem

## Część 2. Porównanie metod wyszukiwania

1. Zestaw testowy: **co najmniej 10 pytań** z oznaczonym dokumentem, w którym jest odpowiedź. Co najmniej 3 pytania to parafrazy (inne słowa niż w dokumencie), co najmniej 2 zawierają nazwę własną, liczbę albo kod.
2. Cztery metody: wektory, BM25, hybryda (RRF), hybryda + reranking cross-encoderem.
3. Tabela: `hit@1`, `hit@3`, `MRR` dla każdej metody.
4. Ta sama tabela (tylko wektory) dla **dwóch innych** ustawień `chunk_size`.
5. W komórce markdown (3–5 zdań): która metoda wypadła najlepiej, na którym pytaniu metody się różniły i dlaczego.

In [ ]:
EVAL = [
    # ("pytanie", "id dokumentu z odpowiedzią", "odpowiedź wzorcowa"),
]

In [ ]:
# TODO: BM25, RRF, reranking, funkcja evaluate i tabela wyników

In [ ]:
# TODO: wpływ chunk_size

*Wnioski:*

## Część 3. RAG i ocena odpowiedzi

1. Funkcja `rag(question)`: wyszukiwanie (dowolna metoda z części 2) → prompt z kontekstem → odpowiedź z identyfikatorami źródeł.
2. Odpowiedzi na **co najmniej 5 pytań** z zestawu testowego oraz **1 pytanie, na które korpus nie zawiera odpowiedzi** (oczekiwane „Nie wiem”).
3. LLM jako sędzia: wierność (faithfulness) i poprawność każdej odpowiedzi, z jednozdaniowym uzasadnieniem, oraz średnie.
4. W komórce markdown (2–4 zdania): czy któraś ocena sędziego jest błędna; jeśli wszystkie są trafne, jaki przypadek mógłby go zmylić.

In [ ]:
# TODO: rag() i odpowiedzi

In [ ]:
# TODO: ocena przez LLM

*Komentarz do ocen sędziego:*